In [ ]:
import os
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.analysis.distances import distance_array
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
import nglview as nv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
wd = "md/results/rbERa_BPA_2"
u = mda.Universe(wd + "/md_cuda.tpr", wd + "/md_fit.xtc")

In [ ]:
last_time_ps = u.trajectory[-1].time
last_time_ns = last_time_ps / 1000

print(f"Trajectory length: {last_time_ns:.2f} ns")

In [ ]:
protein = u.select_atoms("protein")
ligand = u.select_atoms("resname LIG")
complex_sel = u.select_atoms("protein or resname LIG")

print("Protein atoms:", protein.n_atoms)
print("Ligand atoms:", ligand.n_atoms)

In [ ]:
view = nv.show_mdanalysis(u)
view.clear_representations()

view.add_representation("cartoon", selection="protein", color="sstruc")
view.add_representation("ball+stick", selection="LIG", color="element")

view

In [ ]:
R_prot = rms.RMSD(
    u,
    select="protein and backbone",
    groupselections=["protein"],
    ref_frame=0
).run()

time_ns = R_prot.results.rmsd[:,1] / 1000
prot_rmsd = R_prot.results.rmsd[:,2]

In [ ]:
prot_rmsd_df = pd.DataFrame({"Time (ns)": time_ns, "Protein RMSD (Å)": prot_rmsd})
prot_rmsd_df.to_csv(os.path.join(wd, "protein_rmsd.csv"), index=False)

In [ ]:
R_lig = rms.RMSD(
    u,
    select="resname LIG",
    groupselections=["resname LIG"],
    ref_frame=0,
    rmsd_kwargs={"center": True, "superposition": True}
).run()
lig_rmsd = R_lig.results.rmsd[:,2]

In [ ]:
lig_rmsd_df = pd.DataFrame({"Time (ns)": time_ns, "Ligand RMSD (Å)": lig_rmsd})
lig_rmsd_df.to_csv(os.path.join(wd, "ligand_rmsd.csv"), index=False)

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(time_ns, prot_rmsd, label="Protein")
plt.plot(time_ns, lig_rmsd, label="Ligand")

plt.xlabel("Time (ns)")
plt.ylabel("RMSD (Å)")
plt.legend()
plt.title("RMSD vs Time")
plt.tight_layout()
plt.show()

In [ ]:
# Protein–ligand COM distance
dist = []

for ts in u.trajectory:
    d = np.linalg.norm(
        protein.center_of_mass() - ligand.center_of_mass()
    )
    dist.append(d)

plt.plot(time_ns, dist)
plt.xlabel("Time (ns)")
plt.ylabel("COM distance (Å)")
plt.show()

In [ ]:
com_dist_df = pd.DataFrame({"Time (ns)": time_ns, "COM Distance (Å)": dist})
com_dist_df.to_csv(os.path.join(wd, "COM_distance.csv"), index=False)

In [ ]:
# Mean Cα RMSF analysis

# Select Cα atoms only
ca = u.select_atoms("protein and name CA")

# Compute RMSF
RMSF_ca = rms.RMSF(ca).run()

# Per-residue RMSF
ca_rmsf = RMSF_ca.results.rmsf

# Mean Cα RMSF
mean_ca_rmsf = ca_rmsf.mean()
std_ca_rmsf = ca_rmsf.std()

print(f"Mean Cα RMSF: {mean_ca_rmsf:.3f} Å")
print(f"Std  Cα RMSF: {std_ca_rmsf:.3f} Å")

In [ ]:
cd_rmsf_df = pd.DataFrame({"Residue Index": ca.resids, "Cα RMSF (Å)": ca_rmsf})
cd_rmsf_df.to_csv(os.path.join(wd, "CA_RMSF.csv"), index=False)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(ca.residues.resids, ca_rmsf, label="Cα RMSF")
plt.axhline(mean_ca_rmsf, color="red", linestyle="--", label="Mean Cα RMSF")

plt.xlabel("Residue ID")
plt.ylabel("RMSF (Å)")
plt.title("Cα RMSF per Residue")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Align trajectory to protein backbone
align.AlignTraj(
    u,
    u,
    select="protein and backbone",
    in_memory=True
).run()

In [ ]:
# RMSD of Helix 12 (H12): residues 535–547 (222-234 in cleaned PDB)

# Define selection string
h12_sel = "protein and resid 222-234 and name CA"

# Sanity check
print("H12 Cα atoms:", u.select_atoms(h12_sel).n_atoms)

# Compute RMSD
R_h12 = rms.RMSD(
    u,
    u,
    select=h12_sel,
    ref_frame=0
).run()

# Extract data
time_ns = R_h12.results.rmsd[:,1] / 1000  # ps → ns
h12_rmsd = R_h12.results.rmsd[:,2]

# Plot
plt.figure(figsize=(7,4))
plt.plot(time_ns, h12_rmsd)
plt.xlabel("Time (ns)")
plt.ylabel("RMSD (Å)")
plt.title("H12 (Residues 535–547) Cα RMSD")
plt.tight_layout()
plt.show()

In [ ]:
h12_rmsd_df = pd.DataFrame({"Time (ns)": time_ns, "H12 RMSD (Å)": h12_rmsd})
h12_rmsd_df.to_csv(os.path.join(wd, "H12_RMSD.csv"), index=False)

In [ ]:
# Protein residues (shifted numbering)
E40_sel = "protein and resid 40"

# H-bond analysis: Ligand–E40
hb_E40 = HydrogenBondAnalysis(
    universe=u,
    donors_sel=f"(resname LIG) or ({E40_sel})",
    acceptors_sel=f"(resname LIG) or ({E40_sel})"
)

hb_E40.run()

In [ ]:
# Convert H-bond results to DataFrame
cols = ["frame", "donor_idx", "hydrogen_idx", "acceptor_idx", "distance", "angle"]
hb_E40_df = pd.DataFrame(hb_E40.results.hbonds, columns=cols)

# Add time column (ns)
hb_E40_df["time_ns"] = hb_E40_df["frame"] * u.trajectory.dt / 1000

# Save to CSV
hb_E40_df.to_csv(os.path.join(wd, "hbonds_ligand_E40.csv"), index=False)

# Occupancy calculation
frames_with_hb = np.unique(hb_E40.results.hbonds[:, 0])
occupancy = 100 * len(frames_with_hb) / len(u.trajectory)
print(f"Ligand–E40 H-bond occupancy: {occupancy:.1f}%")

In [ ]:
# Protein residues (shifted numbering)
R81_sel = "protein and resid 81"

# H-bond analysis: Ligand–R81
hb_R81 = HydrogenBondAnalysis(
    universe=u,
    donors_sel=f"(resname LIG) or ({R81_sel})",
    acceptors_sel=f"(resname LIG) or ({R81_sel})"
)

hb_R81.run()

In [ ]:
# Convert H-bond results to DataFrame
cols = ["frame", "donor_idx", "hydrogen_idx", "acceptor_idx", "distance", "angle"]
hb_R81_df = pd.DataFrame(hb_R81.results.hbonds, columns=cols)

# Add time column (ns)
hb_R81_df["time_ns"] = hb_R81_df["frame"] * u.trajectory.dt / 1000

# Save to CSV
hb_R81_df.to_csv(os.path.join(wd, "hbonds_ligand_R81.csv"), index=False)

# Occupancy calculation
frames_with_hb = np.unique(hb_R81.results.hbonds[:, 0])
occupancy = 100 * len(frames_with_hb) / len(u.trajectory)
print(f"Ligand–R81 H-bond occupancy: {occupancy:.1f}%")

In [ ]:
# Protein residues (shifted numbering)
M108_sel = "protein and resid 108"

# H-bond analysis: Ligand–M108
hb_M108 = HydrogenBondAnalysis(
    universe=u,
    donors_sel=f"(resname LIG) or ({M108_sel})",
    acceptors_sel=f"(resname LIG) or ({M108_sel})"
)

hb_M108.run()

In [ ]:
# Convert H-bond results to DataFrame
cols = ["frame", "donor_idx", "hydrogen_idx", "acceptor_idx", "distance", "angle"]
hb_M108_df = pd.DataFrame(hb_M108.results.hbonds, columns=cols)

# Add time column (ns)
hb_M108_df["time_ns"] = hb_M108_df["frame"] * u.trajectory.dt / 1000

# Save to CSV
hb_M108_df.to_csv(os.path.join(wd, "hbonds_ligand_M108.csv"), index=False)

# Occupancy calculation
frames_with_hb = np.unique(hb_M108.results.hbonds[:, 0])
occupancy = 100 * len(frames_with_hb) / len(u.trajectory)
print(f"Ligand–M108 H-bond occupancy: {occupancy:.1f}%")

In [ ]:
# Ligand–M108(SD) minimum distance analysis
met_s = u.select_atoms("protein and resid 108 and name SD")
lig_heavy = u.select_atoms("resname LIG and not name H*")

distances = []

for ts in u.trajectory:
    d = distance_array(met_s.positions, lig_heavy.positions).min()
    distances.append(d)

distances = np.array(distances)
time_ns = np.array(range(len(u.trajectory))) * u.trajectory.dt / 1000

min_distance_M108_df = pd.DataFrame({"Time (ns)": time_ns, "Min Distance (Å)": distances})
min_distance_M108_df.to_csv(os.path.join(wd, "min_distance_ligand_M108.csv"), index=False)

plt.figure(figsize=(6,4))
plt.plot(time_ns, distances)
plt.xlabel("Time (ns)")
plt.ylabel("Min distance (Å)")
plt.title("Ligand–M108(SD) Minimum Distance")
plt.tight_layout()
plt.show()

print(f"Mean M108–ligand distance: {distances.mean():.2f} Å")